<a href="https://colab.research.google.com/github/rahiakela/small-language-models-fine-tuning/blob/main/build-and-fine-tune-small-language-model/02_fine_tuning_GPT_2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **How to Build and Fine-Tune a Small Language Model**

*A Step-by-Step Guide for Beginners, Researchers, and Non-Programmers*

1) Paper Version: https://www.amazon.com/dp/B0G3MYWTJK

2) PDF version: https://leanpub.com/howtobuildandfine-tuneasmalllanguagemodel

Reference:

Liu, J. Paul. 2025. How to Build and Fine-Tune a Small Language Model. Leanpub. 479 p.

**Chapter 3, Fine-Tune a Pretrained Model**

Please select a free T4 GPU for this fine-tune:  go to top right, at connect or RAM Disk pull down, click "Change runtime type", see below

## Step 1: Install Required Libraries

In [ ]:
!pip install transformers datasets accelerate -q

print("✓ Libraries installed successfully!")

✓ Libraries installed successfully!


## Step 2: Load a Pre‐trained Model



In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
import torch


####NOTE: you may huggingface token to download the pretrained model

# Load the pretrained GPT-2 model and tokenizer
model_name = "gpt2"  # This is the 125M parameter version

print(f"Loading {model_name}...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model loaded: {model.num_parameters():,} parameters")
print(f"✓ Model size: {model.num_parameters() / 1e6:.0f}M parameters")

Loading gpt2...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✓ Model loaded: 124,439,808 parameters
✓ Model size: 124M parameters


## Step 3: Prepare Your Dataset

In [ ]:
# Sample recipe dataset
recipes = [
    "Preheat oven to 350°F. Mix flour, sugar, and eggs in a large bowl. Pour into greased pan. Bake for 25 minutes until golden brown.",
    "Boil water in a large pot. Add pasta and cook for 8-10 minutes. Drain and toss with olive oil and garlic.",
    "Heat olive oil in a skillet over medium heat. Add diced onions and cook until translucent. Add minced garlic and cook for 1 minute.",
    "Whisk eggs with milk and a pinch of salt. Pour into hot buttered pan. Scramble gently until just set.",
    "Combine ground beef with breadcrumbs, egg, and seasonings. Form into patties. Grill for 5 minutes per side.",
    "Slice vegetables into uniform pieces. Toss with oil and spices. Roast at 400°F for 20-25 minutes, stirring halfway.",
    "Melt butter in saucepan over low heat. Add flour and whisk constantly for 2 minutes. Gradually add milk while whisking.",
    "Season chicken breasts with salt and pepper. Heat oil in pan and cook 6-7 minutes per side until internal temperature reaches 165°F.",
    "Bring broth to a simmer. Add rice and reduce heat to low. Cover and cook for 18 minutes without lifting lid.",
    "Cream butter and sugar until light and fluffy. Add eggs one at a time, beating well after each addition.",
    "Sauté mushrooms in butter until golden. Add white wine and reduce by half. Finish with cream and fresh herbs.",
    "Blanch vegetables in boiling salted water for 2 minutes. Immediately transfer to ice bath to stop cooking.",
    "Dice potatoes into 1-inch cubes. Parboil for 5 minutes, drain, then roast at 425°F until crispy.",
    "Marinate meat in mixture of soy sauce, ginger, and garlic for at least 30 minutes. Pat dry before cooking.",
    "Toast spices in dry pan until fragrant, about 1-2 minutes. Grind in spice grinder or mortar and pestle.",
    "Proof yeast in warm water with a pinch of sugar for 5 minutes until foamy. Mix with flour and knead for 10 minutes.",
    "Sear steak in hot cast iron skillet for 3 minutes per side. Let rest for 5 minutes before slicing against the grain.",
    "Simmer sauce over low heat, stirring occasionally, until reduced and thickened, about 15-20 minutes.",
    "Steam vegetables in bamboo steamer for 5-7 minutes until tender-crisp. Season with sesame oil and soy sauce.",
    "Fold beaten egg whites gently into batter using a rubber spatula. Pour into prepared pan and bake immediately.",
]


# Create more training data by repeating
training_texts = recipes * 10  # 200 total examples

print(f"✓ Dataset prepared: {len(training_texts)} training examples")
print(f"✓ Sample text: {training_texts[0][:80]}...")

✓ Dataset prepared: 200 training examples
✓ Sample text: Preheat oven to 350°F. Mix flour, sugar, and eggs in a large bowl. Pour into gre...


In [ ]:
from datasets import Dataset

# Convert to Hugging Face Dataset format
dataset_dict = {"text": training_texts}
dataset = Dataset.from_dict(dataset_dict)

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,  # Similar to block_size in Chapter 2
        padding="max_length",
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

print("✓ Data tokenized successfully!")
print(f"  Total sequences: {len(tokenized_dataset)}")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✓ Data tokenized successfully!
  Total sequences: 200


In [ ]:
# Set up data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling (like Chapter 2)
)

# Configure training parameters
training_args = TrainingArguments(
    output_dir="./recipe-gpt2",
    overwrite_output_dir=True,
    num_train_epochs=3,  # Just 3 passes through data
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    learning_rate=5e-5,  # Lower than Chapter 2's 3e-4
    logging_steps=50,
    report_to="none",
)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Start training
print("Starting fine-tuning...")
print("This will take about 5-10 minutes on a T4 GPU.\n")

trainer.train()

print("\n✓ Fine-tuning complete!")

Starting fine-tuning...
This will take about 5-10 minutes on a T4 GPU.



Step,Training Loss
50,1.066000
100,0.222800
150,0.137700



✓ Fine-tuning complete!


In [ ]:
# Save the fine-tuned model
model.save_pretrained("./recipe-gpt2-final")
tokenizer.save_pretrained("./recipe-gpt2-final")

print("✓ Model saved to ./recipe-gpt2-final")


✓ Model saved to ./recipe-gpt2-final


In [ ]:
# Function to generate text
def generate_recipe_instruction(prompt, max_length=100):
    """Generate answer based on a prompt"""
    # Encode the prompt
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    # Move to GPU if available
    if torch.cuda.is_available():
        inputs = inputs.to("cuda")
        model.to("cuda")

    # Generate (similar to Chapter 2's generate method)
    outputs = model.generate(
        inputs,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        temperature=0.8,  # Controls randomness
        top_p=0.9,  # Nucleus sampling
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode and return
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# Test with different prompts
prompts = [
    "Heat oil in a pan and",
    "Preheat the oven to",
    "Mix the ingredients in",
    "Season the chicken with",
]



print("Testing the fine-tuned model:\n")
print("=" * 70)

for prompt in prompts:
    result = generate_recipe_instruction(prompt, max_length=80)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {result}")
    print("-" * 70)

Testing the fine-tuned model:


Prompt: Heat oil in a pan and
Generated: Heat oil in a pan and cook 6-7 minutes per side until internal temperature reaches 165°F. Drain and toss with olive oil and garlic. Roast for 10-12 minutes, stirring halfway. Finish with salt and pepper. Season with fresh herbs. Nutrition Information Serving sizes: 1 serving

Calories: 165 calories, 10 grams carbohydrates, 4 carbs, fat 8.7 g
----------------------------------------------------------------------

Prompt: Preheat the oven to
Generated: Preheat the oven to 350°F. Mix flour, sugar, and eggs in a large bowl. Pour into greased pan. Bake for 25 minutes until golden brown. Refrigerate until cheese is golden. Serve immediately.

Tags: baking, grilling, pestle, spice, egg, hot sauce, seasonings, spices, roast, steaming, cooking spices
----------------------------------------------------------------------

Prompt: Mix the ingredients in
Generated: Mix the ingredients in a saucepan over low heat, stirring occ

In [ ]:
# Load the original GPT-2 for comparison
print("Loading original GPT-2 for comparison...\n")
original_model = GPT2LMHeadModel.from_pretrained("gpt2")

def generate_with_original(prompt, max_length=100):
    """Generate text with the original, non-fine-tuned model"""
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = inputs.to("cuda")
        original_model.to("cuda")

    outputs = original_model.generate(
        inputs,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Compare side-by-side
test_prompt = "Heat oil in a pan and"

print("=" * 70)
print(f"Prompt: '{test_prompt}'\n")

original_output = generate_with_original(test_prompt, max_length=80)
finetuned_output = generate_recipe_instruction(test_prompt, max_length=80)

print("ORIGINAL GPT-2:")
print(original_output)
print("\n" + "-" * 70 + "\n")
print("YOUR FINE-TUNED MODEL:")
print(finetuned_output)
print("=" * 70)

Loading original GPT-2 for comparison...

Prompt: 'Heat oil in a pan and'

ORIGINAL GPT-2:
Heat oil in a pan and melt it, add the onions, and cook until softened, about 10 minutes. Add the red pepper flakes and a pinch of salt, bring to a boil, cover and simmer, stirring occasionally. Remove from the heat, let cool for 5 minutes, then remove and set aside.

In a large bowl, combine the diced tomatoes, onion, salt and pepper,

----------------------------------------------------------------------

YOUR FINE-TUNED MODEL:
Heat oil in a pan and cook for 8-10 minutes. Add rice and reduce heat to low. Cover and bake for 18 minutes without lifting lid. Pat dry before cooking. Servings: 18 Calories per serving.

Freestyle cooking with rice: 1
.5 minutes per side. Print Recipe Ingredients 1 cup rice
2 cups sugar
1 tablespoon olive oil
4 cups water



# **Complete Code (All‐in‐One)**

In [ ]:
# Complete Fine-Tuning Script
!pip install transformers datasets accelerate -q

from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import Dataset
import torch

# 1. Load model
print("Loading GPT-2...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. Prepare data
recipes = [
    "Preheat oven to 350°F. Mix flour, sugar, and eggs.",
    "Boil water. Add pasta and cook for 8-10 minutes.",
    "Heat olive oil in a skillet over medium heat. Add diced onions and cook until translucent. Add minced garlic and cook for 1 minute.",
    "Whisk eggs with milk and a pinch of salt. Pour into hot buttered pan. Scramble gently until just set.",
    "Combine ground beef with breadcrumbs, egg, and seasonings. Form into patties. Grill for 5 minutes per side.",
    "Slice vegetables into uniform pieces. Toss with oil and spices. Roast at 400°F for 20-25 minutes, stirring halfway.",
    "Melt butter in saucepan over low heat. Add flour and whisk constantly for 2 minutes. Gradually add milk while whisking.",
    "Season chicken breasts with salt and pepper. Heat oil in pan and cook 6-7 minutes per side until internal temperature reaches 165°F.",

    # ... add your examples
] * 10

dataset = Dataset.from_dict({"text": recipes})
tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True,
                       max_length=128, padding="max_length"),
    batched=True, remove_columns=["text"]
)

# 3. Train
print("Fine-tuning...")
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./model",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        save_steps=500,
        logging_steps=50,
        report_to="none"
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False
    ),
)
trainer.train()

# 4. Test
print("\nTesting...")
model.to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "Heat oil in a pan and"
inputs = tokenizer.encode(prompt, return_tensors="pt")
inputs = inputs.to(model.device)

outputs = model.generate(
    inputs,
    max_length=80,
    temperature=0.8,
    do_sample=True,
    no_repeat_ngram_size=2,
    pad_token_id=tokenizer.eos_token_id
)

print(f"\nPrompt: {prompt}")
print(f"Generated: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
print("\n✓ Done! Your model is fine-tuned.")

Loading GPT-2...


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Fine-tuning...


Step,Training Loss
50,0.845100



Testing...

Prompt: Heat oil in a pan and
Generated: Heat oil in a pan and heat it. Add flour and cook for 1 minute.5 minutes. Gradually add milk while whisking. Scramble gently until just set. Pour into hot water. Refrigerate for at least 20 minutes before serving. Remove from heat. Serve hot. Nutrition Facts Serving size 8 oz (5.0 g) Amount Per Serving: Prep Time: 1 hour

✓ Done! Your model is fine-tuned.
